In [0]:
%pip install plotly==5.24.0
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 109.2 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Not uninstalling plotly at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-1926b0e1-bacd-46ae-a511-b5931dcafc4e
    Can't uninstall 'plotly'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.functions import date_trunc, sum as spark_sum, col

# Read from BDC Delta Share
df = spark.read.table("bdc_share_cash_flow.cashflow.cashflow")

# Aggregate to monthly by company
monthly = (df
    .groupBy(date_trunc("MONTH", col("PostingDate")).alias("month"), "CompanyCode")
    .agg(spark_sum("AmountInGlobalCurrency").alias("cash_flow_eur"))
    .filter(col("month") >= "2024-01-01")
    .orderBy("month", "CompanyCode")
)

# Convert to pandas for plotly (safe for this size after aggregation)
pdf = monthly.toPandas()
print(f"Rows for plotting: {len(pdf)}")
pdf.head()

Rows for plotting: 69


,month,CompanyCode,cash_flow_eur
0,2024-01-01,1010,17682954.0300
1,2024-01-01,1110,-17273.9200
2,2024-01-01,1710,-41987894.1300
3,2024-01-01,AUC1,-5691666.4700
4,2024-01-01,DEC1,-3691684.9700


In [0]:
fig = px.line(
    pdf, 
    x="month", 
    y="cash_flow_eur", 
    color="CompanyCode",
    title="Monthly Cash Flow by Company Code (2024 onwards)",
    labels={"cash_flow_eur": "Cash Flow (EUR)", "month": "Month"},
    markers=True
)

fig.update_layout(
    height=500,
    hovermode="x unified",
    yaxis_tickformat=",.0f"
)

fig.show()

In [0]:
import pandas as pd

# Pivot for heatmap
pivot = pdf.pivot_table(
    index="CompanyCode", 
    columns="month", 
    values="cash_flow_eur",
    aggfunc="sum"
).fillna(0)

fig2 = px.imshow(
    pivot,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Cash Flow Heatmap: Company Code x Month",
    labels={"color": "Cash Flow (EUR)"}
)

fig2.update_layout(height=400)
fig2.show()

In [0]:
top_outflow = (pdf.groupby("CompanyCode")["cash_flow_eur"]
    .sum()
    .sort_values()
    .head(5)
    .reset_index()
)

fig3 = px.bar(
    top_outflow,
    x="CompanyCode",
    y="cash_flow_eur",
    title="Top 5 Company Codes by Total Cash Outflow (2024+)",
    labels={"cash_flow_eur": "Total Cash Flow (EUR)"},
    color="cash_flow_eur",
    color_continuous_scale="Reds_r"
)

fig3.update_layout(height=400, yaxis_tickformat=",.0f")
fig3.show()

In [0]:
top_outflow = (pdf.groupby("CompanyCode")["cash_flow_eur"]
    .sum()
    .loc[lambda x: x < 0]  # only negative = actual outflows
    .sort_values()
    .head(5)
    .reset_index()
)
top_outflow = (pdf.groupby("CompanyCode")["cash_flow_eur"]
    .sum()
    .loc[lambda x: x < 0]
    .sort_values()
    .head(5)
    .reset_index()
)

# Display the dataframe
display(top_outflow)

# Rebuild the chart with corrected data
fig3 = px.bar(
    top_outflow,
    x="CompanyCode",
    y="cash_flow_eur",
    title="Top 5 Company Codes by Total Cash Outflow (2024+)",
    labels={"cash_flow_eur": "Total Cash Flow (EUR)"},
    color="cash_flow_eur",
    color_continuous_scale="Reds_r"
)
fig3.update_layout(height=400, yaxis_tickformat=",.0f")
fig3.show()

In [0]:
top_outflow = (pdf.groupby("CompanyCode")["cash_flow_eur"]
    .sum()
    .loc[lambda x: x < 0]
    .sort_values()
    .head(5)
    .reset_index()
)

# Display the dataframe
display(top_outflow)

# Rebuild the chart with corrected data
fig3 = px.bar(
    top_outflow,
    x="CompanyCode",
    y="cash_flow_eur",
    title="Top 5 Company Codes by Total Cash Outflow (2024+)",
    labels={"cash_flow_eur": "Total Cash Flow (EUR)"},
    color="cash_flow_eur",
    color_continuous_scale="Reds_r"
)
fig3.update_layout(height=400, yaxis_tickformat=",.0f")
fig3.show()

CompanyCode,cash_flow_eur
AUC1,-5691666.4700
USC1,-3922841.0500
DEC1,-3690984.9700
USC2,-470.0200


In [0]:
fig = px.line(
    pdf, 
    x="month", 
    y="cash_flow_eur", 
    color="CompanyCode",
    title="Monthly Cash Flow by Company Code (2024 onwards)",
    labels={"cash_flow_eur": "Cash Flow (EUR)", "month": "Month"},
    markers=True
)
fig.update_layout(height=500, hovermode="x unified", yaxis_tickformat=",.0f")
fig.show()

In [0]:
import pandas as pd

# Pivot for heatmap
pivot = pdf.pivot_table(
    index="CompanyCode", 
    columns="month", 
    values="cash_flow_eur",
    aggfunc="sum"
).fillna(0)

fig2 = px.imshow(
    pivot,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Cash Flow Heatmap: Company Code x Month",
    labels={"color": "Cash Flow (EUR)"}
)

fig2.update_layout(height=400)
fig2.show()

# SAP Finance Insights — Cash Flow Analysis
**Source:** BDC Delta Share `bdc_share_cash_flow.cashflow.cashflow` (SAP S/4HANA)
**Period:** January 2024 onwards
**Volume:** 967K actuals + 793K forecast rows

## Key Findings

1. **Concentration**: Cash flow is dominated by 2 company codes (1710 US, 1010 DE); other entities (1110, 3010, AUC1, DEC1, USC1, USC2) generate near-zero activity across the period
2. **Data quality concern**: Simultaneous collapse of both major companies from mid-2026 onwards suggests a replication/CDC feed slowdown, not a real business event — worth investigating with BDC operators
3. **Seasonal patterns**: Bright peaks visible at Jan 2025 and Jan 2026 for both major companies indicate year-start batch postings
4. **Initial outflow**: Company 1710 shows a significant negative spike (-100M range) in Jan 2024, potentially representing a large payment, dividend, or debt settlement

## Methodology

- Read directly from BDC Delta Share (zero-copy